# Notebook 03: Vehicle Re-ID Training
**Multi-Camera Tracking System — Kaggle Training Pipeline**

Train vehicle re-identification models on VeRi-776 dataset using torchreid.

**Models**: ResNet50-IBN-a (2048-dim) + OSNet-x1.0 (512-dim)  
**Runtime**: GPU T4x2 | **Duration**: ~10-16 hours

## Target Metrics
| Model | mAP | Rank-1 |
|-------|-----|--------|
| ResNet50-IBN-a | ≥60% | ≥85% |
| OSNet-x1.0 | ≥55% | ≥80% |

In [ ]:
!pip install -q torchreid gdown matplotlib pandas

## 1. Setup

In [ ]:
import os
import sys
import json
import time
import shutil
import glob
import re
import os.path as osp
from pathlib import Path
from datetime import datetime

import torch
import numpy as np
import matplotlib.pyplot as plt

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT_DIR = Path("/kaggle/working/reid_models")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# NOTE: root must be WRITABLE — /kaggle/input is read-only
CONFIG = {
    "dataset": "veri776",
    "veri776_root": "/kaggle/working/data",
    "batch_size_train": 64,
    "batch_size_test": 128,
    "num_workers": 2,
    "height": 224,
    "width": 224,
}

print(f"Device: {DEVICE}")

## 2. Dataset Registration

In [ ]:
import torchreid

# ============================================================
# Step 1: Prepare data directory (Kaggle /input is read-only)
# ============================================================
DATA_ROOT = Path("/kaggle/working/data")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

# Kaggle mounts datasets at /kaggle/input/datasets/<owner>/<slug>/
possible_roots = [
    Path("/kaggle/input/datasets/abhyudaya12/veri-vehicle-re-identification-dataset"),
    Path("/kaggle/input/veri-vehicle-re-identification-dataset"),
]

input_dir = None
for root in possible_roots:
    if root.exists():
        input_dir = root
        break

if input_dir is None:
    import subprocess
    result = subprocess.run(["find", "/kaggle/input", "-maxdepth", "5", "-type", "d", "-name", "image_train"],
                            capture_output=True, text=True, timeout=30)
    print(f"Search results:\n{result.stdout}")
    raise RuntimeError(
        f"VeRi dataset not found. Tried: {[str(p) for p in possible_roots]}. "
        "Make sure 'abhyudaya12/veri-vehicle-re-identification-dataset' is attached."
    )

print(f"Dataset found at: {input_dir}")
print("Contents:")
for p in sorted(input_dir.iterdir()):
    kind = f"{len(list(p.iterdir()))} items" if p.is_dir() else "file"
    print(f"  {p.name}/ [{kind}]" if p.is_dir() else f"  {p.name} [{kind}]")

# Find where image_train lives (nested under VeRi/)
veri_data = None
for p in input_dir.rglob("image_train"):
    if p.is_dir():
        veri_data = p.parent
        break

if veri_data is None:
    raise RuntimeError(f"Could not find 'image_train' inside {input_dir}")

print(f"\nVeRi data at: {veri_data}")

# Symlink: /kaggle/working/data/veri -> actual data
symlink = DATA_ROOT / "veri"
if symlink.exists() or symlink.is_symlink():
    symlink.unlink()
symlink.symlink_to(veri_data)

for subdir in ["image_train", "image_query", "image_test"]:
    d = symlink / subdir
    if d.exists():
        n = len(list(d.iterdir()))
        print(f"  {subdir}: {n} files")
    else:
        print(f"  WARNING: {subdir} NOT FOUND!")

# ============================================================
# Step 2: Register custom VeRi dataset (not in pip torchreid)
# ============================================================
class VeRi776(torchreid.data.ImageDataset):
    """VeRi-776 vehicle re-identification dataset.

    Filename format: XXXX_cYYY_ZZZZZZZZ_T.jpg
        XXXX = vehicle ID, cYYY = camera ID
    """
    dataset_dir = 'veri'

    def __init__(self, root='', **kwargs):
        self.root = osp.abspath(osp.expanduser(root))
        self.dataset_dir = osp.join(self.root, self.dataset_dir)

        train_dir = osp.join(self.dataset_dir, 'image_train')
        query_dir = osp.join(self.dataset_dir, 'image_query')
        test_dir = osp.join(self.dataset_dir, 'image_test')

        # relabel=True for train so PIDs are contiguous 0..N-1
        # (required by the classifier head / cross-entropy loss)
        train = self._process_dir(train_dir, relabel=True)
        query = self._process_dir(query_dir, relabel=False)
        gallery = self._process_dir(test_dir, relabel=False)

        super().__init__(train, query, gallery, **kwargs)

    def _process_dir(self, dir_path, relabel=False):
        img_paths = glob.glob(osp.join(dir_path, '*.jpg'))
        if not img_paths:
            sample = os.listdir(dir_path)[:10]
            raise RuntimeError(
                f"No .jpg images in {dir_path}. Sample contents: {sample}"
            )

        pattern = re.compile(r'^(\d+)_c(\d+)')
        data = []
        for img_path in sorted(img_paths):
            fname = osp.basename(img_path)
            match = pattern.match(fname)
            if match:
                pid = int(match.group(1))
                camid = int(match.group(2)) - 1  # zero-index camera IDs
                data.append((img_path, pid, camid))

        if not data:
            sample = [osp.basename(p) for p in img_paths[:5]]
            raise RuntimeError(
                f"No files matched VeRi pattern XXXX_cYYY_... in {dir_path}. "
                f"Sample filenames: {sample}"
            )

        # Relabel PIDs to contiguous 0..N-1 (needed for training set)
        if relabel:
            pid_set = sorted(set(d[1] for d in data))
            pid2label = {pid: label for label, pid in enumerate(pid_set)}
            data = [(img_path, pid2label[pid], camid) for img_path, pid, camid in data]

        n_ids = len(set(d[1] for d in data))
        print(f"  Loaded {osp.basename(dir_path)}: {len(data)} images, {n_ids} IDs (relabel={relabel})")
        return data

torchreid.data.register_image_dataset('veri', VeRi776)
print("\nRegistered custom VeRi-776 dataset")

# ============================================================
# Step 3: Create datamanager
# ============================================================
datamanager = torchreid.data.ImageDataManager(
    root=CONFIG["veri776_root"],
    sources=["veri"],
    targets=["veri"],
    height=CONFIG["height"],
    width=CONFIG["width"],
    batch_size_train=CONFIG["batch_size_train"],
    batch_size_test=CONFIG["batch_size_test"],
    transforms=["random_flip", "random_crop", "random_erasing", "color_jitter"],
    workers=CONFIG["num_workers"],
)

print(f"\nTrain: {datamanager.num_train_pids} vehicle IDs, {datamanager.num_train_cams} cameras")

## 3. Model A: ResNet50-IBN-a Training

**Config**: 120 epochs, Adam lr=3.5e-4, cosine LR schedule, cross-entropy + label smoothing

Note: We use 224×224 input for vehicles (vs 256×128 for persons) since vehicles are more square-shaped.

In [ ]:
model_resnet = torchreid.models.build_model(
    name="resnet50_ibn_a",
    num_classes=datamanager.num_train_pids,
    loss="softmax",
    pretrained=True,
)
model_resnet = model_resnet.to(DEVICE)

# Use both GPUs if available (T4x2)
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs with DataParallel")
    model_resnet = torch.nn.DataParallel(model_resnet)

optimizer_resnet = torchreid.optim.build_optimizer(
    model_resnet, optim="adam", lr=3.5e-4, weight_decay=5e-4,
)
scheduler_resnet = torchreid.optim.build_lr_scheduler(
    optimizer_resnet, lr_scheduler="cosine", max_epoch=120,
)

print(f"Parameters: {sum(p.numel() for p in model_resnet.parameters()):,}")

In [ ]:
engine_resnet = torchreid.engine.ImageSoftmaxEngine(
    datamanager, model_resnet,
    optimizer=optimizer_resnet, scheduler=scheduler_resnet,
    label_smooth=True,
)

print("=" * 60)
print("Training ResNet50-IBN-a on VeRi-776")
print("=" * 60)

start_time = time.time()
engine_resnet.run(
    save_dir=str(OUTPUT_DIR / "vehicle_resnet50_ibn_a"),
    max_epoch=120, eval_freq=20, print_freq=50,
    test_only=False, fixbase_epoch=10,
)
resnet_time = time.time() - start_time
print(f"\nCompleted in {resnet_time / 3600:.1f} hours")

## 4. Evaluate ResNet50-IBN-a

In [ ]:
# Note: engine.run(test_only=True) prints metrics but returns None
engine_resnet.run(
    save_dir=str(OUTPUT_DIR / "vehicle_resnet50_ibn_a"),
    max_epoch=120, test_only=True,
)
print("\nResNet50-IBN-a (Vehicle) evaluation complete. See mAP and Rank-1 metrics above.")

## 5. Model B: OSNet-x1.0 Training

**Config**: 150 epochs, Adam lr=3.5e-4, cosine LR, label smoothing

In [ ]:
model_osnet = torchreid.models.build_model(
    name="osnet_x1_0",
    num_classes=datamanager.num_train_pids,
    loss="softmax",
    pretrained=True,
)
model_osnet = model_osnet.to(DEVICE)

# Use both GPUs if available (T4x2)
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs with DataParallel")
    model_osnet = torch.nn.DataParallel(model_osnet)

optimizer_osnet = torchreid.optim.build_optimizer(
    model_osnet, optim="adam", lr=3.5e-4, weight_decay=5e-4,
)
scheduler_osnet = torchreid.optim.build_lr_scheduler(
    optimizer_osnet, lr_scheduler="cosine", max_epoch=150,
)

print(f"Parameters: {sum(p.numel() for p in model_osnet.parameters()):,}")

In [ ]:
engine_osnet = torchreid.engine.ImageSoftmaxEngine(
    datamanager, model_osnet,
    optimizer=optimizer_osnet, scheduler=scheduler_osnet,
    label_smooth=True,
)

print("=" * 60)
print("Training OSNet-x1.0 on VeRi-776")
print("=" * 60)

start_time = time.time()
engine_osnet.run(
    save_dir=str(OUTPUT_DIR / "vehicle_osnet_x1_0"),
    max_epoch=150, eval_freq=20, print_freq=50,
    test_only=False, fixbase_epoch=10,
)
osnet_time = time.time() - start_time
print(f"\nCompleted in {osnet_time / 3600:.1f} hours")

## 6. Evaluate OSNet-x1.0

In [ ]:
# Note: engine.run(test_only=True) prints metrics but returns None
engine_osnet.run(
    save_dir=str(OUTPUT_DIR / "vehicle_osnet_x1_0"),
    max_epoch=150, test_only=True,
)
print("\nOSNet-x1.0 (Vehicle) evaluation complete. See mAP and Rank-1 metrics above.")

## 7. Results Comparison

In [ ]:
results_summary = {
    "resnet50_ibn_a": {
        "embedding_dim": 2048,
        "input_size": [224, 224],
        "train_time_hours": resnet_time / 3600,
        "epochs": 120,
    },
    "osnet_x1_0": {
        "embedding_dim": 512,
        "input_size": [224, 224],
        "train_time_hours": osnet_time / 3600,
        "epochs": 150,
    },
}

results_path = OUTPUT_DIR / "vehicle_reid_results.json"
with open(results_path, "w") as f:
    json.dump(results_summary, f, indent=2)

print("Vehicle ReID Training Summary")
print("=" * 60)
print(f"{'Model':<20} {'Dim':>5} {'Epochs':>7} {'Time (h)':>9}")
print("-" * 60)
for name, info in results_summary.items():
    print(f"{name:<20} {info['embedding_dim']:>5} {info['epochs']:>7} {info['train_time_hours']:>9.1f}")

## 8. Export Models

In [ ]:
export_dir = Path("/kaggle/working/exported_models")
export_dir.mkdir(parents=True, exist_ok=True)

models_to_export = {
    "vehicle_resnet50_ibn_a_veri776.pth": OUTPUT_DIR / "vehicle_resnet50_ibn_a" / "model" / "model-best.pth.tar",
    "vehicle_osnet_x1_0_veri776.pth": OUTPUT_DIR / "vehicle_osnet_x1_0" / "model" / "model-best.pth.tar",
}

for export_name, src_path in models_to_export.items():
    if src_path.exists():
        dst = export_dir / export_name
        shutil.copy2(src_path, dst)
        print(f"Exported: {export_name} ({dst.stat().st_size / 1024**2:.1f} MB)")
    else:
        # Try epoch-specific checkpoint
        epoch = 120 if "resnet" in export_name else 150
        alt = src_path.parent / f"model.pth.tar-{epoch}"
        if alt.exists():
            shutil.copy2(alt, export_dir / export_name)
            print(f"Exported (alt): {export_name}")
        else:
            print(f"Warning: {src_path} not found")

metadata = {
    "task": "vehicle_reid",
    "dataset": "veri776",
    "models": {
        "resnet50_ibn_a": {
            "architecture": "resnet50_ibn_a",
            "embedding_dim": 2048,
            "input_size": [224, 224],
            "file": "vehicle_resnet50_ibn_a_veri776.pth",
        },
        "osnet_x1_0": {
            "architecture": "osnet_x1_0",
            "embedding_dim": 512,
            "input_size": [224, 224],
            "file": "vehicle_osnet_x1_0_veri776.pth",
        },
    },
    "trained_at": datetime.now().isoformat(),
}

with open(export_dir / "vehicle_reid_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(f"\nDownload from: {export_dir}")
print("Place in local project: models/reid/")

## Next Steps
1. **Download** exported models from `/kaggle/working/exported_models/`
2. Place in local project: `models/reid/`
3. Both person (Notebook 02) and vehicle models are now ready
4. Optionally proceed to **Notebook 04** for TransReID (stretch goal)

### Models Summary
After running Notebooks 02 and 03, you should have:
- `person_osnet_x1_0_market1501.pth` — Person ReID (512-dim)
- `person_resnet50_ibn_a_market1501.pth` — Person ReID (2048-dim)
- `vehicle_osnet_x1_0_veri776.pth` — Vehicle ReID (512-dim)
- `vehicle_resnet50_ibn_a_veri776.pth` — Vehicle ReID (2048-dim)